# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eman123-123/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one pseudonymized content item for one pseudonymized client on one report date.

**Time window:** I will use the March 2026 warehouse slice for development and verification. Features will use information available at the decision moment, while the future outcome/label will be kept separate to avoid leakage.

In [21]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
import pandas as pd

# Create DuckDB connection
con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [22]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [23]:
# Connect to the FlyRank internship warehouse
REL = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse connection ready.")

Warehouse connection ready.


In [24]:
# Check the warehouse files available for the March 2026 panel
con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
### Five features

1. `gsc_impressions` — available at the decision moment because it is observed Search Console performance up to that reporting date.
2. `gsc_clicks` — available at the decision moment because it is observed Search Console click performance up to that reporting date.
3. `gsc_sum_position` — available at the decision moment because it summarizes observed Search Console position up to that reporting date.
4. `sessions_ai` — available at the decision moment because it records observed AI-referred sessions for that reporting period.
5. `scroll_events` — available at the decision moment because it records observed engagement events for that reporting period.

### Label

`future_gsc_clicks` — a future-period outcome used only as the label, not as an input feature.

### Context

`report_date`, `client_hash_id`, `content_hash_id`, `gsc_data_available`, and `ga4_data_available`.

### Excluded

Query-level fields, client-identifying information, URLs, private queries, and other identifying information are excluded from the feature frame.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [26]:
# Query 1 — Verify the grain
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_date_client_content
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_date_client_content
0,9841378,9841378


In [27]:
con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [28]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [29]:
# Five-feature frame: March 2026 features -> April 2026 outcome
# March is the decision period; April provides the future label.

march = """
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        sessions_ai,
        scroll_events
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
"""

april = """
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS future_gsc_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
"""

feature_df = con.sql(f"""
WITH march AS ({march}),
april AS ({april})
SELECT
    m.client_hash_id,
    m.content_hash_id,
    COALESCE(m.gsc_impressions, 0) AS gsc_impressions,
    COALESCE(m.gsc_clicks, 0) AS gsc_clicks,
    COALESCE(m.gsc_sum_position, 0) AS gsc_sum_position,
    COALESCE(m.sessions_ai, 0) AS sessions_ai,
    COALESCE(m.scroll_events, 0) AS scroll_events,
    CASE WHEN COALESCE(a.future_gsc_clicks, 0) > 0
         THEN 1 ELSE 0 END AS label,
    COALESCE(a.future_gsc_clicks, 0) AS future_gsc_clicks
FROM march m
LEFT JOIN april a
    USING (client_hash_id, content_hash_id)
""").df()

print("Feature frame shape:", feature_df.shape)
feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (9841378, 9)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,sessions_ai,scroll_events,label,future_gsc_clicks
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67,0,0,1,2.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0,0,0,0,0.0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616,0,0,1,8.0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28,0,0,0,0.0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25,0,0,1,2.0


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "sessions_ai",
    "scroll_events"
]

X = feature_df[features].fillna(0)
y = feature_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

honest_score = roc_auc_score(
    y_test,
    model.predict_proba(X_test)[:, 1]
)

# Deliberate leakage: give the model the future outcome itself.
X_leak = X.copy()
X_leak["LEAK_future_gsc_clicks"] = feature_df["future_gsc_clicks"]

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y, test_size=0.25, random_state=42, stratify=y
)

leak_model = LogisticRegression(max_iter=1000)
leak_model.fit(Xl_train, yl_train)

leak_score = roc_auc_score(
    yl_test,
    leak_model.predict_proba(Xl_test)[:, 1]
)

print("Honest AUC:", round(honest_score, 4))
print("Leaked AUC:", round(leak_score, 4))

Honest AUC: 0.8982
Leaked AUC: 1.0


In [31]:
# Remove the deliberately leaked column.
X_clean = feature_df[features].fillna(0)

print("Final feature columns:")
for col in features:
    print("-", col)

print("\nLeaked column removed: LEAK_future_gsc_clicks")
print("Honest AUC kept:", round(honest_score, 4))

Final feature columns:
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- sessions_ai
- scroll_events

Leaked column removed: LEAK_future_gsc_clicks
Honest AUC kept: 0.8982


### Leakage lesson

I deliberately added `future_gsc_clicks` as a feature even though it is the future outcome being predicted. The quick score increased sharply, demonstrating why label-derived information must not be available to the model at decision time. I removed the leaked column and kept the honest score from the five decision-time features.

**Honest score:** the AUC reported above after removing the leaked feature.

**Limitation:** This is a directional leakage demonstration, not evidence of causal performance or production model quality.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This warehouse is an unbalanced panel, so clients do not necessarily have the same history length. Some rows have GSC data available while others do not, and GA4 availability is also uneven. In March 2026, 3,611,061 of 9,841,378 rows had GSC data available and 413,966 had GA4 data available, using `IS TRUE`.

The March slice therefore cannot be treated as complete coverage of all measurement systems. Different reporting and lookback windows can also overlap, so a row should not automatically be interpreted as an independent causal observation. The data can support measured and directional decision-support analysis, but it cannot by itself establish causal effects or explain why a page performed differently.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.